In [1]:
import sys

In [2]:
import json

In [3]:
sys.path.append("../src")

In [4]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace

c:\Users\andre\anaconda3\envs\kag_env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# REBEL

In [15]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


In [5]:
import torch
print(torch.cuda.is_available())  # True / False
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


## PRUEBA

In [18]:

def extract_triplets(text):
    triplets = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
    return triplets


In [ ]:
# Load model and tokenizer


In [14]:
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large").to("cuda")
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 3,
    "num_return_sequences": 3,
}


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 4104.73it/s]


In [ ]:
# Text to extract triplets from
text = 'Punta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.'

# Tokenizer text
# model_inputs = tokenizer(text, max_length=256, padding=True, truncation=True, return_tensors = 'pt')
model_inputs = tokenizer(text, max_length=256, padding=True, truncation=True, return_tensors = 'pt').to("cuda")


In [11]:
# Generate
generated_tokens = model.generate(
    model_inputs["input_ids"].to(model.device),
    attention_mask = model_inputs["attention_mask"].to(model.device),
    **gen_kwargs,
)


In [16]:
generated_tokens = model.generate(
    model_inputs["input_ids"].to("cuda"),
    attention_mask = model_inputs["attention_mask"].to("cuda"),
    **gen_kwargs,
)

In [17]:
# Extract text
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

# Extract triplets
for idx, sentence in enumerate(decoded_preds):
    print(f'Prediction triplets sentence {idx}')
    print(extract_triplets(sentence))

Prediction triplets sentence 0
[{'head': 'Punta Cana', 'type': 'located in the administrative territorial entity', 'tail': 'La Altagracia Province'}, {'head': 'Punta Cana', 'type': 'country', 'tail': 'Dominican Republic'}, {'head': 'Higüey', 'type': 'located in the administrative territorial entity', 'tail': 'La Altagracia Province'}, {'head': 'Higüey', 'type': 'country', 'tail': 'Dominican Republic'}, {'head': 'La Altagracia Province', 'type': 'country', 'tail': 'Dominican Republic'}, {'head': 'Dominican Republic', 'type': 'contains administrative territorial entity', 'tail': 'La Altagracia Province'}]
Prediction triplets sentence 1
[{'head': 'Punta Cana', 'type': 'located in the administrative territorial entity', 'tail': 'La Altagracia Province'}, {'head': 'Higüey', 'type': 'located in the administrative territorial entity', 'tail': 'La Altagracia Province'}, {'head': 'Higüey', 'type': 'country', 'tail': 'Dominican Republic'}, {'head': 'La Altagracia Province', 'type': 'country', 't

In [8]:
decoded_preds

['<s><triplet> Punta Cana <subj> La Altagracia Province <obj> located in the administrative territorial entity <subj> Dominican Republic <obj> country <triplet> Higüey <subj> La Altagracia Province <obj> located in the administrative territorial entity <subj> Dominican Republic <obj> country <triplet> La Altagracia Province <subj> Dominican Republic <obj> country <triplet> Dominican Republic <subj> La Altagracia Province <obj> contains administrative territorial entity</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>',
 '<s><triplet> Punta Cana <subj> La Altagracia Province <obj> located in the administrative territorial entity <triplet> Higüey <subj> La Altagracia Province <obj> located in the administrative territorial entity <subj> Dominican Republic <obj> country <triplet> La Altagracia Province <subj> Dominican Republic <obj> country <triplet> Dominican Republic <subj> La Altagracia Province <obj> contains administrative territorial entity</

## PRUEBA CON 2Wiki

In [16]:
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large").to("cuda")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 3523.76it/s]


In [6]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 10, "train")

In [ ]:
dataset_2Wiki[0]

In [21]:
json.loads(dataset_2Wiki[0]['context'])

[['Stuart Rosenberg',
  ['Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984).',
   'He was noted for his work with actor Paul Newman.']],
 ['Méditerranée (1963 film)',
  ['Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff.',
   'It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel.',
   'The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year.',
   'Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table

In [8]:
frases = [txt[1] for txt in json.loads(dataset_2Wiki[0]['context']) ]
frases

[['Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984).',
  'He was noted for his work with actor Paul Newman.'],
 ['Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff.',
  'It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel.',
  'The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year.',
  'Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.'],
 ['Move is a 1970 American comedy film starring Elliot

In [25]:
for frase in (frases[0]):
    print(frase)


Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984).
He was noted for his work with actor Paul Newman.


In [10]:
# Text to extract triplets from
text = " ".join(frases[0])
text


'Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984). He was noted for his work with actor Paul Newman.'

In [49]:
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 8,
    "num_return_sequences": 8,
}
# Tokenizer text
model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors = 'pt').to("cuda")

# Generate
generated_tokens = model.generate(
    # model_inputs["input_ids"].to(model.device),
    # attention_mask=model_inputs["attention_mask"].to(model.device),
    model_inputs["input_ids"].to("cuda"),
    attention_mask=model_inputs["attention_mask"].to("cuda"),
    **gen_kwargs,
)

# Extract text
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

triplets = {}
# Extract triplets
for idx, sentence in enumerate(decoded_preds):
    print(f'Prediction triplets sentence {idx}')
    tripletas = extract_triplets(sentence)
    triplets[idx] = tripletas
    print(tripletas)
    

Prediction triplets sentence 0
[{'head': 'Stuart Rosenberg', 'type': 'date of birth', 'tail': 'August 11, 1927'}, {'head': 'Stuart Rosenberg', 'type': 'date of death', 'tail': 'March 15, 2007'}, {'head': 'Cool Hand Luke', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'Voyage of the Damned', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Amityville Horror', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Pope of Greenwich Village', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Pope of Greenwich Village', 'type': 'cast member', 'tail': 'Paul Newman'}]
Prediction triplets sentence 1
[{'head': 'Stuart Rosenberg', 'type': 'date of birth', 'tail': 'August 11, 1927'}, {'head': 'Stuart Rosenberg', 'type': 'date of death', 'tail': 'March 15, 2007'}, {'head': 'Cool Hand Luke', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'Voyage of the Damned', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Amityville Horror

In [23]:
for ev in json.loads(dataset_2Wiki[0]["evidences"]):
    print(ev)

['Move (1970 film)', 'director', 'Stuart Rosenberg']
['Méditerranée (1963 film)', 'director', 'Jean-Daniel Pollet']
['Stuart Rosenberg', 'country of citizenship', 'American']
['Jean-Daniel Pollet', 'country of citizenship', 'French']


In [50]:
for k in triplets:
    for trip in triplets[k]:
        print(f"{trip['head']} ->  {trip['type']}  -> {trip['tail']}")
    print("-"*30)
    

Stuart Rosenberg ->  date of birth  -> August 11, 1927
Stuart Rosenberg ->  date of death  -> March 15, 2007
Cool Hand Luke ->  director  -> Stuart Rosenberg
Voyage of the Damned ->  director  -> Stuart Rosenberg
The Amityville Horror ->  director  -> Stuart Rosenberg
The Pope of Greenwich Village ->  director  -> Stuart Rosenberg
The Pope of Greenwich Village ->  cast member  -> Paul Newman
------------------------------
Stuart Rosenberg ->  date of birth  -> August 11, 1927
Stuart Rosenberg ->  date of death  -> March 15, 2007
Cool Hand Luke ->  director  -> Stuart Rosenberg
Voyage of the Damned ->  director  -> Stuart Rosenberg
The Amityville Horror ->  director  -> Stuart Rosenberg
The Pope of Greenwich Village ->  director  -> Stuart Rosenberg
------------------------------
Stuart Rosenberg ->  date of birth  -> August 11, 1927
Stuart Rosenberg ->  date of death  -> March 15, 2007
Cool Hand Luke ->  director  -> Stuart Rosenberg
Voyage of the Damned ->  director  -> Stuart Rosenbe

In [47]:
for k in triplets:
    for trip in triplets[k]:
        print(f"{trip['head']} ->  {trip['type']}  -> {trip['tail']}")
    print("-"*30)

Stuart Rosenberg ->  date of birth  -> August 11, 1927
Stuart Rosenberg ->  date of death  -> March 15, 2007
Cool Hand Luke ->  director  -> Stuart Rosenberg
Voyage of the Damned ->  director  -> Stuart Rosenberg
The Amityville Horror ->  director  -> Stuart Rosenberg
The Pope of Greenwich Village ->  director  -> Stuart Rosenberg
The Pope of Greenwich Village ->  cast member  -> Paul Newman
------------------------------
Stuart Rosenberg ->  date of birth  -> August 11, 1927
Stuart Rosenberg ->  date of death  -> March 15, 2007
Cool Hand Luke ->  director  -> Stuart Rosenberg
Voyage of the Damned ->  director  -> Stuart Rosenberg
The Amityville Horror ->  director  -> Stuart Rosenberg
The Pope of Greenwich Village ->  director  -> Stuart Rosenberg
------------------------------
Stuart Rosenberg ->  date of birth  -> August 11, 1927
Stuart Rosenberg ->  date of death  -> March 15, 2007
Cool Hand Luke ->  director  -> Stuart Rosenberg
Voyage of the Damned ->  director  -> Stuart Rosenbe

In [33]:
for frase in (frases[8]):
    print(frase)


Jean-Daniel Pollet (1936–2004) was a French film director and screenwriter who was most active in the 1960s and 1970s.
He was associated with two approaches to filmmaking: comedies which blended burlesque and melancholic elements, and poetic films based on texts by writers such as the French poet Francis Ponge.


In [51]:
text = " ".join(frases[1])
text

'Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.'

In [35]:
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 8,
    "num_return_sequences": 8,
}
# Tokenizer text
model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors = 'pt').to("cuda")

# Generate
generated_tokens = model.generate(
    # model_inputs["input_ids"].to(model.device),
    # attention_mask=model_inputs["attention_mask"].to(model.device),
    model_inputs["input_ids"].to("cuda"),
    attention_mask=model_inputs["attention_mask"].to("cuda"),
    **gen_kwargs,
)

# Extract text
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

triplets = {}
# Extract triplets
for idx, sentence in enumerate(decoded_preds):
    print(f'Prediction triplets sentence {idx}')
    tripletas = extract_triplets(sentence)
    triplets[idx] = tripletas
    print(tripletas)

Prediction triplets sentence 0
[{'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'film director'}, {'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'screenwriter'}]
Prediction triplets sentence 1
[{'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'film director'}]
Prediction triplets sentence 2
[{'head': 'Jean-Daniel Pollet', 'type': 'country of citizenship', 'tail': 'French'}]
Prediction triplets sentence 3
[{'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'screenwriter'}]
Prediction triplets sentence 4
[{'head': 'Jean-Daniel Pollet', 'type': 'country of citizenship', 'tail': 'French'}, {'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'film director'}, {'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'screenwriter'}]
Prediction triplets sentence 5
[{'head': 'Jean-Daniel Pollet', 'type': 'country of citizenship', 'tail': 'French'}, {'head': 'Jean-Daniel Pollet', 'type': 'occupation', 'tail': 'film director'}]
Predic

In [37]:
trip_tuplas = []
for k in triplets:
    for trip in triplets[k]:
        print(f"{trip['head']} ->  {trip['type']}  -> {trip['tail']}")
        tup = (trip['head'], trip['type'], trip['tail'])
        trip_tuplas.append(tup)
    print("-"*30)

Jean-Daniel Pollet ->  occupation  -> film director
Jean-Daniel Pollet ->  occupation  -> screenwriter
------------------------------
Jean-Daniel Pollet ->  occupation  -> film director
------------------------------
Jean-Daniel Pollet ->  country of citizenship  -> French
------------------------------
Jean-Daniel Pollet ->  occupation  -> screenwriter
------------------------------
Jean-Daniel Pollet ->  country of citizenship  -> French
Jean-Daniel Pollet ->  occupation  -> film director
Jean-Daniel Pollet ->  occupation  -> screenwriter
------------------------------
Jean-Daniel Pollet ->  country of citizenship  -> French
Jean-Daniel Pollet ->  occupation  -> film director
------------------------------
Francis Ponge ->  country of citizenship  -> French
------------------------------
Jean-Daniel Pollet ->  country of citizenship  -> French
Jean-Daniel Pollet ->  occupation  -> screenwriter
------------------------------


In [40]:
list(set(trip_tuplas))

[('Jean-Daniel Pollet', 'occupation', 'film director'),
 ('Jean-Daniel Pollet', 'country of citizenship', 'French'),
 ('Francis Ponge', 'country of citizenship', 'French'),
 ('Jean-Daniel Pollet', 'occupation', 'screenwriter')]

## PRUEBA PASANDOLE UNA LISTA DE TEXTOS

EMPEORA - TARDA MAS!

In [56]:
frases

[['Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984).',
  'He was noted for his work with actor Paul Newman.'],
 ['Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff.',
  'It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel.',
  'The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year.',
  'Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.'],
 ['Move is a 1970 American comedy film starring Elliot

In [57]:
texts = [" ".join(frase) for frase in frases]

In [59]:
texts

['Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984). He was noted for his work with actor Paul Newman.',
 'Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.',
 'Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss a

In [ ]:
text = " ".join(frases[0])
text


In [ ]:
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 8,
    "num_return_sequences": 8,
}

model_inputs = tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors = 'pt').to("cuda")

# Generate
generated_tokens = model.generate(
    # model_inputs["input_ids"].to(model.device),
    # attention_mask=model_inputs["attention_mask"].to(model.device),
    model_inputs["input_ids"].to("cuda"),
    attention_mask=model_inputs["attention_mask"].to("cuda"),
    **gen_kwargs,
)

# Extract text
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)


In [68]:
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 8,
    "num_return_sequences": 8,
}

for text in texts:
    model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors = 'pt').to("cuda")

    # Generate
    generated_tokens = model.generate(
        # model_inputs["input_ids"].to(model.device),
        # attention_mask=model_inputs["attention_mask"].to(model.device),
        model_inputs["input_ids"].to("cuda"),
        attention_mask=model_inputs["attention_mask"].to("cuda"),
        **gen_kwargs,
    )

    # Extract text
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

In [70]:
print(model.device)

cuda:0


In [71]:
print(model_inputs["input_ids"].device)

cuda:0


In [61]:
decoded_preds

['<s><triplet> Stuart Rosenberg <subj> August 11, 1927 <obj> date of birth <subj> March 15, 2007 <obj> date of death <triplet> Cool Hand Luke <subj> Stuart Rosenberg <obj> director <triplet> Voyage of the Damned <subj> Stuart Rosenberg <obj> director <triplet> The Amityville Horror <subj> Stuart Rosenberg <obj> director <triplet> The Pope of Greenwich Village <subj> Stuart Rosenberg <obj> director <subj> Paul Newman <obj> cast member</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>',
 '<s><triplet> Stuart Rosenberg <subj> August 11, 1927 <obj> date of birth <subj> March 15, 2007 <obj> date of death <triplet> Cool Hand Luke <subj> Stuart Rosenberg <obj> director <triplet> Voyage of the Damned <subj> Stuart Rosenberg <obj> director <triplet> The Amityville Horror <subj> Stuart Rosenberg <obj> director <triplet> The Pope of Greenwich Village <sub

In [62]:

triplets = {}
tripletas_tup_list = []
for idx, sentence in enumerate(decoded_preds):
    print(f'Prediction triplets sentence {idx}')
    tripletas = extract_triplets(sentence)
    tripletas_tup = extract_triplets_tuples(sentence)
    # tripletas_tup_list.append(tripletas_tup)
    tripletas_tup_list += tripletas_tup
    triplets[idx] = tripletas
    print(tripletas)

Prediction triplets sentence 0
[{'head': 'Stuart Rosenberg', 'type': 'date of birth', 'tail': 'August 11, 1927'}, {'head': 'Stuart Rosenberg', 'type': 'date of death', 'tail': 'March 15, 2007'}, {'head': 'Cool Hand Luke', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'Voyage of the Damned', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Amityville Horror', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Pope of Greenwich Village', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Pope of Greenwich Village', 'type': 'cast member', 'tail': 'Paul Newman'}]
Prediction triplets sentence 1
[{'head': 'Stuart Rosenberg', 'type': 'date of birth', 'tail': 'August 11, 1927'}, {'head': 'Stuart Rosenberg', 'type': 'date of death', 'tail': 'March 15, 2007'}, {'head': 'Cool Hand Luke', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'Voyage of the Damned', 'type': 'director', 'tail': 'Stuart Rosenberg'}, {'head': 'The Amityville Horror

In [64]:
tripletas_tup_list

[('Stuart Rosenberg', 'date of birth', 'August 11, 1927'),
 ('Stuart Rosenberg', 'date of death', 'March 15, 2007'),
 ('Cool Hand Luke', 'director', 'Stuart Rosenberg'),
 ('Voyage of the Damned', 'director', 'Stuart Rosenberg'),
 ('The Amityville Horror', 'director', 'Stuart Rosenberg'),
 ('The Pope of Greenwich Village', 'director', 'Stuart Rosenberg'),
 ('The Pope of Greenwich Village', 'cast member', 'Paul Newman'),
 ('Stuart Rosenberg', 'date of birth', 'August 11, 1927'),
 ('Stuart Rosenberg', 'date of death', 'March 15, 2007'),
 ('Cool Hand Luke', 'director', 'Stuart Rosenberg'),
 ('Voyage of the Damned', 'director', 'Stuart Rosenberg'),
 ('The Amityville Horror', 'director', 'Stuart Rosenberg'),
 ('The Pope of Greenwich Village', 'director', 'Stuart Rosenberg'),
 ('Stuart Rosenberg', 'date of birth', 'August 11, 1927'),
 ('Stuart Rosenberg', 'date of death', 'March 15, 2007'),
 ('Cool Hand Luke', 'director', 'Stuart Rosenberg'),
 ('Voyage of the Damned', 'director', 'Stuart Rose

In [65]:
list(set(tripletas_tup_list))

[('Contempt', 'director', 'Jean-Luc Goddard'),
 ('Winchel Koch', 'date of birth', 'April 11, 1916'),
 ('Joel Lieber', 'work period (start)', '1970'),
 ('Ian Barry', 'occupation', 'film and TV director'),
 ('Stuart Rosenberg', 'date of birth', 'August 11, 1927'),
 ('Peter Levin', 'field of work', 'film'),
 ('Move', 'cast member', 'Elliott Gould'),
 ('Ian Barry', 'occupation', 'director of film and TV'),
 ('The Pope of Greenwich Village', 'country of origin', 'American'),
 ('Voyage of the Damned', 'cast member', 'Paul Newman'),
 ('Peter Levin', 'occupation', 'film'),
 ('Peter Levin', 'occupation', 'director of film'),
 ('Cool Hand Luke', 'country of origin', 'American'),
 ('Stuart Rosenberg', 'work period (start)', '1970'),
 ('Peter Levin', 'occupation', 'director'),
 ('Breda', 'country', 'Netherlands'),
 ('Brian Johnson', 'country of citizenship', 'British'),
 ('Méditerranée', 'producer', 'Barbet Schroeder'),
 ('Jean-Daniel Pollet', 'country of citizenship', 'French'),
 ('Ian Barry', 'c

## FUNCION EXTRACCION

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large").to("cuda")

In [ ]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 10, "train")

In [ ]:
def extract_triplets(text):
    triplets = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
    return triplets

In [127]:
def extract_triplets_tuples(text):
    triplets = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                if subject != object_:
                    triplets.append((subject.strip(), relation.strip().replace(" ", "_"), object_.strip()))
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                if subject != object_:
                    triplets.append((subject.strip(), relation.strip().replace(" ", "_"), object_.strip()))
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        if subject != object_:
            triplets.append((subject.strip(), relation.strip().replace(" ", "_"), object_.strip()))
    return triplets

In [102]:
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 8,
    "num_return_sequences": 8,
}

model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors = 'pt').to("cuda")

# Generate
generated_tokens = model.generate(
    # model_inputs["input_ids"].to(model.device),
    # attention_mask=model_inputs["attention_mask"].to(model.device),
    model_inputs["input_ids"].to("cuda"),
    attention_mask=model_inputs["attention_mask"].to("cuda"),
    **gen_kwargs,
)

# Extract text
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

# triplets = {}
tripletas_tup_list = []
for idx, sentence in enumerate(decoded_preds):
    # print(f'Prediction triplets sentence {idx}')
    # tripletas = extract_triplets(sentence)
    tripletas_tup = extract_triplets_tuples(sentence)
    # tripletas_tup_list.append(tripletas_tup)
    tripletas_tup_list += tripletas_tup
    # triplets[idx] = tripletas
    # print(tripletas)

In [103]:
tripletas_tup_list

[('Howard Winchel Koch', 'date_of_birth', 'April 11, 1916'),
 ('Howard Winchel Koch', 'date_of_death', 'February 16, 2001'),
 ('Winchel Koch', 'date_of_birth', 'April 11, 1916'),
 ('Winchel Koch', 'date_of_death', 'February 16, 2001'),
 ('Howard Winchel Koch', 'date_of_birth', 'April 11, 1916'),
 ('Howard Winchel Koch', 'date_of_death', 'February 16, 2001'),
 ('Howard Winchel Koch', 'country_of_citizenship', 'American'),
 ('Winchel Koch', 'date_of_birth', 'April 11, 1916'),
 ('Winchel Koch', 'date_of_death', 'February 16, 2001'),
 ('Winchel Koch', 'country_of_citizenship', 'American'),
 ('Howard Winchel Koch', 'date_of_birth', 'April 11, 1916'),
 ('Winchel Koch', 'date_of_birth', 'April 11, 1916'),
 ('Howard Winchel Koch', 'date_of_death', 'February 16, 2001'),
 ('Winchel Koch', 'date_of_death', 'February 16, 2001')]

In [80]:
list(set(tripletas_tup_list))

[('Winchel Koch', 'date of birth', 'April 11, 1916'),
 ('Howard Winchel Koch', 'date of birth', 'April 11, 1916'),
 ('Howard Winchel Koch', 'date of death', 'February 16, 2001'),
 ('Howard Winchel Koch', 'country of citizenship', 'American'),
 ('Winchel Koch', 'date of death', 'February 16, 2001'),
 ('Winchel Koch', 'country of citizenship', 'American')]

In [128]:
def extraer_tripletas_rebel(text):
    gen_kwargs = {
        "max_length": 256,
        "length_penalty": 0,
        "num_beams": 8,
        "num_return_sequences": 8,
    }

    model_inputs = tokenizer(text, max_length=512, padding=True, truncation=True, return_tensors = 'pt').to("cuda")

    # Generate
    generated_tokens = model.generate(
        # model_inputs["input_ids"].to(model.device),
        # attention_mask=model_inputs["attention_mask"].to(model.device),
        model_inputs["input_ids"].to("cuda"),
        attention_mask=model_inputs["attention_mask"].to("cuda"),
        **gen_kwargs,
    )

    # Extract text
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

    # triplets = {}
    tripletas_tup_list = []
    for sentence in decoded_preds:
        # print(f'Prediction triplets sentence {idx}')
        # tripletas = extract_triplets(sentence)
        tripletas_tup = extract_triplets_tuples(sentence)
        # tripletas_tup_list.append(tripletas_tup)
        tripletas_tup_list += tripletas_tup
        # triplets[idx] = tripletas
        # print(tripletas)
    return list(set(tripletas_tup_list))

In [129]:
parrafos = [" ".join(txt[1]) for txt in json.loads(dataset_2Wiki[0]['context']) ]
parrafos

['Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984). He was noted for his work with actor Paul Newman.',
 'Méditerranée is a 1963 French experimental film directed by Jean-Daniel Pollet with assistance from Volker Schlöndorff. It was written by Philippe Sollers and produced by Barbet Schroeder, with music by Antione Duhamel. The 45 minute film is cited as one of Pollet\'s most influential films, which according to Jonathan Rosenbaum directly influenced Jean-Luc Goddard\'s "Contempt", released later the same year. Footage for the film was shot around the Mediterranean, including at a Greek temple, a Sicilian garden, the sea, and also features a fisherman, a bullfighter, and a girl on an operating table.',
 'Move is a 1970 American comedy film starring Elliott Gould, Paula Prentiss a

In [134]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 100, "train")

In [ ]:
tripletas_all = []
for reg in dataset_2Wiki:
    parrafos = [" ".join(txt[1]) for txt in json.loads(reg['context']) ]
    for par in parrafos:
        tripletas = extraer_tripletas_rebel(par)
        tripletas_all += tripletas

In [136]:
tripletas_all

[('Stuart Rosenberg', 'date_of_birth', 'August 11, 1927'),
 ('Stuart Rosenberg', 'country_of_citizenship', 'American'),
 ('The Pope of Greenwich Village', 'cast_member', 'Paul Newman'),
 ('The Amityville Horror', 'country_of_origin', 'American'),
 ('Stuart Rosenberg', 'date_of_death', 'March 15, 2007'),
 ('The Pope of Greenwich Village', 'country_of_origin', 'American'),
 ('Cool Hand Luke', 'country_of_origin', 'American'),
 ('Cool Hand Luke', 'cast_member', 'Paul Newman'),
 ('The Pope of Greenwich Village', 'director', 'Stuart Rosenberg'),
 ('Voyage of the Damned', 'country_of_origin', 'American'),
 ('The Amityville Horror', 'director', 'Stuart Rosenberg'),
 ('Cool Hand Luke', 'director', 'Stuart Rosenberg'),
 ('Voyage of the Damned', 'director', 'Stuart Rosenberg'),
 ('The Amityville Horror', 'cast_member', 'Paul Newman'),
 ('Voyage of the Damned', 'cast_member', 'Paul Newman'),
 ('Contempt', 'director', 'Jean-Luc Goddard'),
 ('Méditerranée', 'producer', 'Barbet Schroeder'),
 ('Médit

## INSERTAR EN NEO4J

In [107]:
from neo4j import GraphDatabase

In [121]:
# def insertar_triplets_batch(self, tripletas):
driver = GraphDatabase.driver(
            "bolt://localhost:7687",
            auth=("neo4j", "password"),
            database = "2wiki.prueba.rebel"
        )
def insertar_triplets_batch(driver, tripletas):
    
    """
    Inserta tripletas en Neo4j en batch.
    input: tripletas -> lista de tuplas (subj, rel, obj)

    """
    
    query_base = f"""
        UNWIND $tripletas as tripleta
        MERGE (a:Entity {{name: tripleta[0]}})
        MERGE (b:Entity {{name: tripleta[2]}})
        WITH a, b, tripleta
        CALL apoc.create.relationship(a, tripleta[1], {{}}, b)
        YIELD rel
        RETURN rel;
    """
    # summary = self.driver.execute_query(
    summary = driver.execute_query(
        query_base,
        tripletas = tripletas,
        # database_ = self.database
    )
    return summary


In [137]:
tripletas_all = list(set(tripletas_all))

In [138]:
summary = insertar_triplets_batch(driver, tripletas_all)

In [13]:
query_base = """
            UNWIND $tripletas as tripleta
            MERGE (a:Entity {name: tripleta[0]})
            MERGE (b:Entity {name: tripleta[2]})
            WITH a, b, tripleta
            CALL apoc.create.relationship(a, tripleta[1], {}, b)
            YIELD rel
            RETURN rel;
        """

In [125]:
tripletas_all

[('Roy William Neill', 'employer', 'Universal Studios'),
 ('Sherlock Holmes films', 'distributed_by', 'Universal Studios'),
 ('Israeli Academy of Film and Television', 'country', 'Israel'),
 ('vihuela', 'subclass_of', 'string instrument'),
 ('Leave It To Me', 'instance_of', 'song'),
 ('Warsaw Pact invasion of Czechoslovakia', 'point_in_time', '1968'),
 ('Charge It', 'director', 'Harry Garson'),
 ('The Falcon Takes Over', 'genre', 'mystery film'),
 ('Engal Aasan', 'cast_member', 'Vijayakanth'),
 ('Leave It To Me', 'has_part', 'Leave It To Me'),
 ('La légion saute sur Kolwezi', 'narrative_location', 'French Guiana'),
 ('Walter Ulfig', 'occupation', 'composer of film scores'),
 ('Suleiman the Conqueror', 'publication_date', '1961'),
 ('Leave It to Me', 'publication_date', '1955'),
 ("Stoltenberg's Second Cabinet", 'country', 'Norwegian'),
 ('Thomas Morse', 'country_of_citizenship', 'American'),
 ('Weekend in Paradise', 'cast_member', 'Julius Falkenstein'),
 ('Abe Meyer', 'country_of_citiz

# OpenIE

Con libreria StanfordOpenIE

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from openie import StanfordOpenIE


In [2]:
text = "Barack Obama was born in Hawaii."

with StanfordOpenIE() as client:
    for triple in client.annotate(text):
        print('Triple:', triple)

Starting server with command: java -Xmx8G -cp C:\Users\andre\.stanfordnlp_resources\stanford-corenlp-4.5.3/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9000 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-b420247125224ebf.props -preload openie


PermanentlyFailedException: Timed out waiting for service to come alive.

# Spacy

In [3]:
import spacy

In [7]:
# from triplet_extract import TripletExtractor
from triplet_extract import extract

In [2]:
import triplet_extract

In [10]:
import spacy
from triplet_extract import extract 


nlp = spacy.load("en_core_web_sm") 

texto = "Messi won the world cup."
tripletas = extract(texto) 

for t in tripletas:
    print(t)


(Messi, won, the world cup)
(Messi, won, world cup)


In [12]:
txt = 'Stuart Rosenberg (August 11, 1927 – March 15, 2007) was an American film and television director whose motion pictures include "Cool Hand Luke" (1967), "Voyage of the Damned" (1976), "The Amityville Horror" (1979), and "The Pope of Greenwich Village" (1984). He was noted for his work with actor Paul Newman.'
tripletas = extract(txt) 

for t in tripletas:
    print(t)

(whose motion pictures, include, Cool Hand Luke " ( 1967))
(Stuart Rosenberg ( August 11, 1927–March 15, 2007), was, an American film and television director whose motion pictures include " Cool Hand Luke " ( 1967))
(whose motion pictures, include, Cool Hand Luke ")
(Stuart Rosenberg August, 1927 March, 2007), was, American film and television director whose motion pictures include " Cool Hand Luke " ( 1967))
(Stuart Rosenberg August, 1927,), was, American film and television director whose motion pictures include " Cool Hand Luke " ( 1967))
(whose pictures, include, Cool Hand Luke ")
(whose pictures, include, Cool Hand Luke ( 1967))
(He, was noted for, his work with actor Paul Newman)
(whose pictures, include, Cool Hand Luke ()
